In [ ]:
import pandas as pd

In [14]:
df = pd.read_csv("../data/financial_fraud_detection_dataset.csv")
df.head()

,transaction_id,timestamp,sender_account,receiver_account,amount,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,time_since_last_transaction,spending_deviation_score,velocity_score,geo_anomaly_score,payment_channel,ip_address,device_hash
0,T100000,2023-08-22T09:22:43.516168,ACC877572,ACC388389,343.78,withdrawal,utilities,Tokyo,mobile,False,NaN,NaN,-0.21,3,0.22,card,13.101.214.112,D8536477
1,T100001,2023-08-04T01:58:02.606711,ACC895667,ACC944962,419.65,withdrawal,online,Toronto,atm,False,NaN,NaN,-0.14,7,0.96,ACH,172.52.47.194,D2622631
2,T100002,2023-05-12T11:39:33.742963,ACC733052,ACC377370,2773.86,deposit,other,London,pos,False,NaN,NaN,-1.78,20,0.89,card,185.98.35.23,D4823498
3,T100003,2023-10-10T06:04:43.195112,ACC996865,ACC344098,1666.22,deposit,online,Sydney,pos,False,NaN,NaN,-0.60,6,0.37,wire_transfer,107.136.36.87,D9961380
4,T100004,2023-09-24T08:09:02.700162,ACC584714,ACC497887,24.43,transfer,utilities,Toronto,mobile,False,NaN,NaN,0.79,13,0.27,ACH,108.161.108.255,D7637601


In [15]:
print("=" * 40)
print("SUMMARY DATASET TRANSACTIONS")
print("=" * 40)

print(f"\nTotal {len(df)} transactions")
print(f"Total {len(df.columns)} columns")

SUMMARY DATASET TRANSACTIONS

Total 5000000 transactions
Total 18 columns


---

# Step 1: Data Cleaning
### Kiểm tra và xử lý Missing Values & Duplicates

In [16]:
# Kiểm tra Missing Values
print("=" * 40)
print("MISSING VALUES ANALYSIS")
print("=" * 40)
missing_info = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

print(missing_info[missing_info['Missing_Count'] > 0])
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")
print(f"Total Duplicate Rows: {df.duplicated().sum()}")

MISSING VALUES ANALYSIS
                                                  Column  Missing_Count  \
fraud_type                                    fraud_type        4820447   
time_since_last_transaction  time_since_last_transaction         896513   

                             Missing_Percentage  
fraud_type                                96.41  
time_since_last_transaction               17.93  

Total Missing Values: 5716960
Total Duplicate Rows: 0


In [17]:
# Xử lý Missing Values
print("\n" + "=" * 40)
print("HANDLING MISSING VALUES")
print("=" * 40)

# Xử lý fraud_type: Khi is_fraud = False thì fraud_type = 'None'
df['fraud_type'] = df['fraud_type'].fillna('Not Fraud')
print("fill fraud_type from NaN => 'Not Fraud'")

# Xóa các missing value của cột time_since_last_transaction (không thể fill)
df_cleaned = df.dropna()
print(f"Rows removed due to NaN: {len(df) - len(df_cleaned)}")

df = df_cleaned
print(f"Final dataset shape: {df.shape}")


HANDLING MISSING VALUES
fill fraud_type from NaN => 'Not Fraud'
Rows removed due to NaN: 896513
Final dataset shape: (4103487, 18)


---

# Step 2: Type Casting
### Chuyển đổi các cột sang đúng kiểu dữ liệu

In [18]:
print("\n" + "=" * 40)
print("TYPE CASTING - CONVERTING COLUMNS TYPES")
print("=" * 40)

# Datetime Columns
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed')
print("1. timestamp => datetime")

# Numeric (Float) Columns
numeric_columns = [
    'amount', 'time_since_last_transaction',
    'spending_deviation_score', 'velocity_score', 'geo_anomaly_score'
]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
print(f"2. Numeric => float")

# String/ID Columns
string_columns = [
    'transaction_id', 'sender_account', 'receiver_account', 'ip_address', 'device_hash'
]
for col in string_columns:
    if col in df.columns:
        df[col] = df[col].astype('string')
print(f"3. Object => string")

# Category Columns
category_columns = [
    'transaction_type', 'merchant_category', 'location', 'device_used',
    'payment_channel', 'fraud_type'
]
for col in category_columns:
    if col in df.columns:
        df[col] = df[col].astype('category')
print(f"4. Category columns => string to category")

# Boolean Columns
df['is_fraud'] = df['is_fraud'].astype('bool')
print("5. is_fraud => bool")

print("\nData Types after Casting:")
print(df.dtypes)


TYPE CASTING - CONVERTING COLUMNS TYPES
1. timestamp => datetime
2. Numeric => float
3. Object => string
4. Category columns => string to category
5. is_fraud => bool

Data Types after Casting:
transaction_id                 string[python]
timestamp                      datetime64[ns]
sender_account                 string[python]
receiver_account               string[python]
amount                                float64
transaction_type                     category
merchant_category                    category
location                             category
device_used                          category
is_fraud                                 bool
fraud_type                           category
time_since_last_transaction           float64
spending_deviation_score              float64
velocity_score                        float64
geo_anomaly_score                     float64
payment_channel                      category
ip_address                     string[python]
device_hash            

---

# Step 3: Feature Engineering
### Trích xuất thông tin thời gian từ timestamp

In [19]:
print("\n" + "=" * 40)
print("FEATURE ENGINEERING")
print("=" * 40)

# Trích xuất features từ timestamp (cho table dim_date)

# Transaction Date
df['transaction_date'] = df['timestamp'].dt.date
print("1. transaction_date (Date only)")

# Hour of Transaction
df['hour'] = df['timestamp'].dt.hour
print("2. hour (0-23)")

# Day of Week (0=Monday, 6=Sunday)
df['day_of_week'] = df['timestamp'].dt.dayofweek
print("3. day_of_week (0=Monday, 6=Sunday)")

# Is Weekend (True if Saturday or Sunday)
df['is_weekend'] = df['day_of_week'].isin([5, 6])
print("4. is_weekend (Boolean)")

print(f"\n=> New columns created: transaction_date, hour, day_of_week, is_weekend")
print(f"\nSample of new features:")
df[['timestamp', 'transaction_date', 'hour', 'day_of_week', 'is_weekend']].head(3)


FEATURE ENGINEERING
1. transaction_date (Date only)
2. hour (0-23)
3. day_of_week (0=Monday, 6=Sunday)
4. is_weekend (Boolean)

=> New columns created: transaction_date, hour, day_of_week, is_weekend

Sample of new features:


,timestamp,transaction_date,hour,day_of_week,is_weekend
154,2023-04-25 14:30:25.650085,2023-04-25,14,1,False
3369,2023-08-17 01:49:02.932894,2023-08-17,1,3,False
4242,2023-12-28 23:55:39.204732,2023-12-28,23,3,False


---

# Step 4: Star Schema Transformation
### Tách dữ liệu thành Dimension Tables & Fact Table
#### Foreign Key Relationships: transaction_date → date_id, merchant_category → merchant_id, location → location_id, device → device_id, accounts → account_id

In [20]:
print("\n" + "=" * 40)
print("STAR SCHEMA TRANSFORMATION")
print("=" * 40)

# ============================================================================
# DIM_ACCOUNTS: Bảng dimension cho các tài khoản
# ============================================================================
print("\nCreating dim_accounts table")

accounts_sender = df[['sender_account']].rename(columns={'sender_account': 'account_id'}).drop_duplicates()
accounts_receiver = df[['receiver_account']].rename(columns={'receiver_account': 'account_id'}).drop_duplicates()

# Concat and drop_duplicates
dim_accounts = pd.concat([accounts_sender, accounts_receiver], ignore_index=True).drop_duplicates().reset_index(drop=True)
dim_accounts = dim_accounts.sort_values('account_id').reset_index(drop=True)

print(f"Total unique accounts: {len(dim_accounts)}")
print(f"Columns: {list(dim_accounts.columns)}")

# ============================================================================
# DIM_MERCHANTS: Bảng dimension cho các danh mục thương mại
# ============================================================================
print("\nCreating dim_merchants table")

dim_merchants = df[['merchant_category']].drop_duplicates().reset_index(drop=True)
dim_merchants = dim_merchants.sort_values('merchant_category').reset_index(drop=True)
dim_merchants.insert(0, 'merchant_id', range(1, len(dim_merchants) + 1))
print(f"Total unique merchant categories: {len(dim_merchants)}")

# ============================================================================
# DIM_LOCATIONS: Bảng dimension cho các địa điểm
# ============================================================================
print("\nCreating dim_location table")

dim_locations = df[['location']].drop_duplicates().reset_index(drop=True)
dim_locations = dim_locations.sort_values('location').reset_index(drop=True)

# Add location_id
dim_locations.insert(0, 'location_id', range(1, len(dim_locations) + 1))

print(f"Total unique locations: {len(dim_locations)}")
print(f"Columns: {list(dim_locations.columns)}")

# ============================================================================
# 📱 DIM_DEVICES: Bảng dimension cho các thiết bị
# ============================================================================
print("\nCreating dim_devices table")

dim_devices = df[['device_used', 'device_hash', 'ip_address']].drop_duplicates().reset_index(drop=True)
dim_devices = dim_devices.sort_values(['device_used', 'device_hash']).reset_index(drop=True)

# Add device_id
dim_devices.insert(0, 'device_id', range(1, len(dim_devices) + 1))
print(f"Total unique device combinations: {len(dim_devices)}")
print(f"Columns: {list(dim_devices.columns)}")

# ============================================================================
# DIM_DATE: Bảng dimension cho thời gian
# ============================================================================
print("\nCreating dim_date table")

dim_date = df[['transaction_date', 'day_of_week', 'is_weekend']].drop_duplicates().reset_index(drop=True)
dim_date = dim_date.sort_values('transaction_date').reset_index(drop=True)
dim_date['date_id'] = dim_date['transaction_date'].astype(str)

# Thêm thông tin ngày, tháng, năm
dim_date['day'] = pd.to_datetime(dim_date['transaction_date']).dt.day
dim_date['month'] = pd.to_datetime(dim_date['transaction_date']).dt.month
dim_date['year'] = pd.to_datetime(dim_date['transaction_date']).dt.year

# Sắp xếp lại cột
dim_date = dim_date[['date_id', 'transaction_date', 'year', 'month', 'day', 'day_of_week', 'is_weekend']]
print(f"Total unique dates: {len(dim_date)}")
print(f"Columns: {list(dim_date.columns)}")


STAR SCHEMA TRANSFORMATION

Creating dim_accounts table
Total unique accounts: 899790
Columns: ['account_id']

Creating dim_merchants table
Total unique merchant categories: 8

Creating dim_location table
Total unique locations: 8
Columns: ['location_id', 'location']

Creating dim_devices table
Total unique device combinations: 4103487
Columns: ['device_id', 'device_used', 'device_hash', 'ip_address']

Creating dim_date table
Total unique dates: 366
Columns: ['date_id', 'transaction_date', 'year', 'month', 'day', 'day_of_week', 'is_weekend']


In [21]:
# ============================================================================
# FACT_TRANSACTIONS: Bảng Fact chính chứa các giao dịch
# ============================================================================
print("\nCreating fact_transactions table")

fact_transactions = df[['transaction_id', 'sender_account', 'receiver_account',
                        'transaction_date', 'hour', 'merchant_category',
                        'location', 'device_used', 'device_hash', 'ip_address',
                        'amount', 'time_since_last_transaction',
                        'spending_deviation_score', 'velocity_score', 'geo_anomaly_score',
                        'payment_channel', 'is_fraud', 'fraud_type']].copy()

# Mapping: sender_account → sender_account_id
fact_transactions = fact_transactions.merge(
    dim_accounts.reset_index().rename(columns={'account_id': 'sender_account', 'index': 'sender_account_id'}),
    on='sender_account', how='left'
).drop('sender_account', axis=1)

# Mapping: receiver_account → receiver_account_id
fact_transactions = fact_transactions.merge(
    dim_accounts.reset_index().rename(columns={'account_id': 'receiver_account', 'index': 'receiver_account_id'}),
    on='receiver_account', how='left'
).drop('receiver_account', axis=1)

# Mapping: merchant_category → merchant_id
if 'merchant_category' in fact_transactions.columns:
    fact_transactions = fact_transactions.merge(
        dim_merchants[['merchant_id', 'merchant_category']],
        on='merchant_category', how='left'
    ).drop('merchant_category', axis=1)

# Mapping: location → location_id
if 'location' in fact_transactions.columns:
    fact_transactions = fact_transactions.merge(
        dim_locations[['location_id', 'location']],
        on='location', how='left'
    ).drop('location', axis=1)

# Mapping: device_used + device_hash + ip_address → device_id
if 'device_used' in fact_transactions.columns:
    fact_transactions = fact_transactions.merge(
        dim_devices[['device_id', 'device_used', 'device_hash', 'ip_address']],
        on=['device_used', 'device_hash', 'ip_address'], how='left'
    ).drop(['device_used', 'device_hash', 'ip_address'], axis=1)

# Mapping: transaction_date → date_id
fact_transactions['date_id'] = fact_transactions['transaction_date'].astype(str)
fact_transactions = fact_transactions.merge(
    dim_date[['date_id']],
    on='date_id', how='left'
).drop('transaction_date', axis=1)

# Sắp xếp lại cột: transaction_id, foreign keys, measures
fact_transactions = fact_transactions[[
    'transaction_id',
    'sender_account_id', 'receiver_account_id',
    'date_id', 'hour',
    'merchant_id', 'location_id', 'device_id',
    'amount', 'time_since_last_transaction',
    'spending_deviation_score', 'velocity_score', 'geo_anomaly_score',  
    'payment_channel', 'is_fraud', 'fraud_type'
]]

print(f"Total transactions in fact table: {fact_transactions.shape}")
print(f"Columns: {list(fact_transactions.columns)}")


Creating fact_transactions table
Total transactions in fact table: (4103487, 16)
Columns: ['transaction_id', 'sender_account_id', 'receiver_account_id', 'date_id', 'hour', 'merchant_id', 'location_id', 'device_id', 'amount', 'time_since_last_transaction', 'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'payment_channel', 'is_fraud', 'fraud_type']


---

# Step 5: Verification & Summary
### Xem cấu trúc các bảng Dimension và Fact

In [24]:
print("\n" + "=" * 40)
print("STAR SCHEMA SUMMARY & STATISTICS")
print("=" * 40)

# Thống kê
fraud_stats = fact_transactions['is_fraud'].value_counts()
fraud_pct = (fraud_stats / len(fact_transactions) * 100).round(2)

print("\nFRAUD STATISTICS IN FACT TABLE:")
print(f"   - Legitimate Transactions: {fraud_stats.get(False, 0):,} ({fraud_pct.get(False, 0):}%)")
print(f"   - Fraudulent Transactions: {fraud_stats.get(True, 0):,} ({fraud_pct.get(True, 0):}%)")
print(f"   - Total Transactions: {len(fact_transactions):,}")


STAR SCHEMA SUMMARY & STATISTICS

FRAUD STATISTICS IN FACT TABLE:
   - Legitimate Transactions: 3,923,934 (95.62%)
   - Fraudulent Transactions: 179,553 (4.38%)
   - Total Transactions: 4,103,487


In [25]:
print("=" * 80)
print("FACT_TRANSACTIONS TABLE - DATA STRUCTURE & SAMPLE")
print("=" * 80)
print(f"\nTotal Transactions: {len(fact_transactions)}")
print("\nSample Data (First 3 rows):")              # fact_transactions
print(fact_transactions.head(3).to_string())

print("\n\n" + "=" * 80)
print("DIM_ACCOUNTS TABLE - SAMPLE & STRUCTURE")      # dim_account
print("=" * 80)
print(f"\nTotal Accounts: {len(dim_accounts)}")
print("\nSample Data (First 5 rows):")
print(dim_accounts.head(5).to_string(index=False))

print("\n\n" + "=" * 80)
print("DIM_MERCHANTS TABLE - SAMPLE & STRUCTURE")     # dim_merchants
print("=" * 80)

print(f"\nTotal Merchant Categories: {len(dim_merchants)}")
print("\nSample Data (First 5 rows):")
print(dim_merchants.head(5).to_string(index=False))

print("\n\n" + "=" * 80)
print("DIM_LOCATIONS TABLE - SAMPLE & STRUCTURE")     # dim_location
print("=" * 80)
print(f"\nTotal Locations: {len(dim_locations)}")
print("\nSample Data (First 5 rows):")
print(dim_locations.head(5).to_string(index=False))

print("\n\n" + "=" * 80)
print("DIM_DEVICES TABLE - SAMPLE & STRUCTURE")     # dim_device
print("=" * 80)
print(f"\nTotal Device Combinations: {len(dim_devices)}")
print("\nSample Data (First 3 rows):")
print(dim_devices.head(3).to_string(index=False))

print("\n\n" + "=" * 80)
print("DIM_DATE TABLE - SAMPLE & STRUCTURE")    # dim_date
print("=" * 80)
print(f"\nTotal Date-Hour Combinations: {len(dim_date)}")
print("\nSample Data (First 5 rows):")
print(dim_date.head(5).to_string(index=False))

FACT_TRANSACTIONS TABLE - DATA STRUCTURE & SAMPLE

Total Transactions: 4103487

Sample Data (First 3 rows):
  transaction_id  sender_account_id  receiver_account_id     date_id  hour  merchant_id  location_id  device_id  amount  time_since_last_transaction  spending_deviation_score  velocity_score  geo_anomaly_score payment_channel  is_fraud fraud_type
0        T100154             320138               122607  2023-04-25    14            5            7    2320854  318.12                 -4797.552868                     -0.94            16.0               0.64             UPI     False  Not Fraud
1        T103369             659704               333793  2023-08-17     1            3            2    2502131   25.03                  3705.738348                     -0.56             1.0               0.48             ACH     False  Not Fraud
2        T104242             602094               558460  2023-12-28    23            3            8    2996085    5.33                  2158.906433   

---

# Step 6: Save dataset
### Lưu lại dataset dưới dạng csv

In [ ]:
# fact_transactions = fact_transactions.to_csv("../data/fact_transactions.csv", index=False)
# dim_accounts = dim_accounts.to_csv("../data/dim_accounts.csv", index=False)
# dim_merchants = dim_merchants.to_csv("../data/dim_merchants.csv", index=False)
# dim_locations = dim_locations.to_csv("../data/dim_locations.csv", index=False)
# dim_devices = dim_devices.to_csv("../data/dim_devices.csv", index=False)
# dim_date = dim_date.to_csv("../data/dim_date.csv", index=False)